# nnU-Net v2 Training — Vast.ai

Trains nnU-Net v2 to segment the gross tumour volume (GTV) on our **own**
NSCLC-Radiomics cohort (RTSTRUCT-derived masks as ground truth). No external
checkpoints, no fine-tuning — nnU-Net self-configures and trains from our data.

## Compute plan (Vast.ai)
- **GPU:** rent a single 24 GB+ card (RTX 4090 or A100 recommended); `-device cuda`.
- **Disk:** resize the instance to **≥ 100 GB** before starting (raw DICOM + NIfTI + preprocessed).
- **Trainer:** `nnUNetTrainer_250epochs` (cost-controlled first pass). Switch `TRAINER`
  to `nnUNetTrainer` for the full 1000-epoch run once the pipeline is validated.
- **Wall-clock estimate:** download ~45 min · convert ~30 min · plan+preprocess ~25 min ·
  **train fold 0 ~1.5–3 h** on a 4090 → ~3–4.5 h total, a few USD of GPU time.
- Training checkpoints every 50 epochs; a disconnect is resumable with `--c` (section 8).

Run the cells top to bottom. Section 4 requires a one-time file upload.

## 1 — Check GPU and disk

In [ ]:
import subprocess, shutil

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True)
print('GPU:', result.stdout.strip() or '(nvidia-smi failed — is this a GPU instance?)')

total, used, free = shutil.disk_usage('/')
print(f'Disk: {free / 1e9:.0f} GB free / {total / 1e9:.0f} GB total')
assert free / 1e9 > 100, 'Need at least 100 GB free — resize the disk on Vast.ai first'

## 2 — Install dependencies

In [ ]:
# %pip installs into the *running kernel* — the same interpreter the convert
# subprocess uses (sys.executable). Avoids the %%bash/!pip wrong-env trap.
%pip install -q nnunetv2 pydicom rt_utils SimpleITK tcia_utils pandas tqdm
print('Dependencies installed into:', __import__('sys').executable)

## 3 — Configure paths and nnU-Net environment variables

nnU-Net v2 reads **`nnUNet_raw`**, **`nnUNet_preprocessed`**, **`nnUNet_results`**
(exact names). We set them in `os.environ` so the later `%%bash` cells inherit them.

In [ ]:
import os
import sys
from pathlib import Path

WORKSPACE      = Path('/workspace')
RAW_DIR        = WORKSPACE / 'data' / 'raw'          # downloaded DICOMs
NNUNET_RAW     = WORKSPACE / 'nnunet' / 'raw'        # nnU-Net raw datasets
NNUNET_PREP    = WORKSPACE / 'nnunet' / 'preprocessed'
NNUNET_RESULTS = WORKSPACE / 'nnunet' / 'results'

for d in [RAW_DIR, NNUNET_RAW, NNUNET_PREP, NNUNET_RESULTS]:
    d.mkdir(parents=True, exist_ok=True)

# nnU-Net v2 official env var names
os.environ['nnUNet_raw']          = str(NNUNET_RAW)
os.environ['nnUNet_preprocessed'] = str(NNUNET_PREP)
os.environ['nnUNet_results']      = str(NNUNET_RESULTS)

# Convenience vars for %%bash cells
os.environ['RAW_DIR']    = str(RAW_DIR)
os.environ['DATASET_ID'] = '001'

# Trainer variant — controls epochs (and Vast.ai cost). Folder name below must match.
TRAINER = 'nnUNetTrainer_250epochs'
os.environ['TRAINER'] = TRAINER
CONFIG  = '3d_fullres'
DATASET = 'Dataset001_NSCLCRadiomics'

print('nnU-Net env configured:')
for k in ['nnUNet_raw', 'nnUNet_preprocessed', 'nnUNet_results']:
    print(f'  {k} = {os.environ[k]}')
print(f'  TRAINER = {TRAINER}  |  CONFIG = {CONFIG}')
# Ensure %%bash cells can find the nnU-Net CLI scripts (they live in
# the kernel env's bin, which a bare bash shell may not have on PATH).
os.environ['PATH'] = os.path.dirname(sys.executable) + os.pathsep + os.environ['PATH']


## 4 — Upload the conversion script

Upload **`scripts/preprocessing/prepare_nnunet_dataset.py`** into this working
directory using the Jupyter file browser (drag-and-drop). The next cell checks it.

In [ ]:
PREPARE_SCRIPT = Path('prepare_nnunet_dataset.py')
assert PREPARE_SCRIPT.exists(), (
    'Upload prepare_nnunet_dataset.py into this directory first '
    '(Jupyter file browser → Upload).')
print('Found', PREPARE_SCRIPT.resolve())

## 5 — Download NSCLC-Radiomics from TCIA  (~45 min)

Downloads every CT and RTSTRUCT series in the collection. The converter groups
series by PatientID from the DICOM headers, so the exact folder layout tcia_utils
produces does not matter.

In [ ]:
from tcia_utils import nbia
import pandas as pd

COLLECTION = 'NSCLC-Radiomics'
print('Fetching series list from TCIA...')
# getSeries returns a list of dicts — wrap it in a DataFrame for filtering
series_df = pd.DataFrame(nbia.getSeries(collection=COLLECTION))
print(f'Total series: {len(series_df)}')
print(series_df['Modality'].value_counts().to_string())

In [ ]:
# Download by SeriesInstanceUID lists (input_type='list' is the stable API).
# The converter groups DICOMs by PatientID from headers, so download layout doesn't matter.
ct_uids = series_df.loc[series_df['Modality'] == 'CT', 'SeriesInstanceUID'].tolist()
rt_uids = series_df.loc[series_df['Modality'] == 'RTSTRUCT', 'SeriesInstanceUID'].tolist()

print(f'Downloading {len(ct_uids)} CT series...')
nbia.downloadSeries(ct_uids, input_type='list', path=str(RAW_DIR))

print(f'Downloading {len(rt_uids)} RTSTRUCT series...')
nbia.downloadSeries(rt_uids, input_type='list', path=str(RAW_DIR))

n_dcm = sum(1 for _ in RAW_DIR.rglob('*.dcm'))
print(f'\nDownloaded DICOM files under {RAW_DIR}: {n_dcm}')
assert n_dcm > 0, 'No DICOMs downloaded — check TCIA connectivity'

## 6 — Convert DICOM → nnU-Net format  (~30 min)

In [ ]:
import subprocess, sys

proc = subprocess.run(
    [sys.executable, str(PREPARE_SCRIPT),
     '--raw-dir', str(RAW_DIR),
     '--out-dir', str(NNUNET_RAW),
     '--workers', '8'],
    text=True, capture_output=True)
print(proc.stdout[-4000:])
if proc.returncode != 0:
    print('STDERR:', proc.stderr[-4000:])
    raise RuntimeError('Conversion failed')

In [ ]:
import json
dataset_dir = NNUNET_RAW / DATASET
images = sorted((dataset_dir / 'imagesTr').glob('*.nii.gz'))
labels = sorted((dataset_dir / 'labelsTr').glob('*.nii.gz'))
print(f'imagesTr: {len(images)}  |  labelsTr: {len(labels)}')
assert len(images) > 0, 'No images produced — inspect conversion output above'
assert len(images) == len(labels), f'image/label mismatch: {len(images)} vs {len(labels)}'
with open(dataset_dir / 'dataset.json') as f:
    print('dataset.json:', json.dumps(json.load(f), indent=2))

## 7 — Plan and preprocess  (~25 min)

nnU-Net reads the dataset fingerprint and auto-configures patch size, batch size,
spacing, and network depth. `--verify_dataset_integrity` catches CT/mask geometry
mismatches before training.

In [ ]:
%%bash
nnUNetv2_plan_and_preprocess -d "$DATASET_ID" --verify_dataset_integrity -c 3d_fullres

## 8 — Train fold 0 (3d_fullres)

`nnUNetTrainer_250epochs` on a single 4090 ≈ 1.5–3 h. Checkpoints every 50 epochs.
**Walk away once this starts.** If the session drops, run the resume cell below.

In [ ]:
%%bash
nnUNetv2_train "$DATASET_ID" 3d_fullres 0 -tr "$TRAINER" --val_best -device cuda

### Resume if the session disconnected

In [ ]:
# Run this ONLY to resume after a disconnect (adds --c to continue from latest checkpoint).
# %%bash
# nnUNetv2_train "$DATASET_ID" 3d_fullres 0 -tr "$TRAINER" --val_best -device cuda --c

## 9 — Inspect validation DSC

In [ ]:
import json
model_folder = (NNUNET_RESULTS / DATASET /
                f'{TRAINER}__nnUNetPlans__{CONFIG}' / 'fold_0')
summary_path = model_folder / 'validation' / 'summary.json'
if summary_path.exists():
    summary = json.load(open(summary_path))
    print(f"Validation DSC (fold 0): {summary['foreground_mean']['Dice']:.4f}")
else:
    print('summary.json not found yet. Files present:')
    for p in sorted(model_folder.rglob('*')):
        print(' ', p.relative_to(model_folder))

## 10 — Package model for download

In [ ]:
import shutil
model_dir = NNUNET_RESULTS / DATASET / f'{TRAINER}__nnUNetPlans__{CONFIG}'
out_zip = WORKSPACE / f'nnunet_model_{DATASET}'
shutil.make_archive(str(out_zip), 'zip', root_dir=str(NNUNET_RESULTS), base_dir=DATASET)
zip_path = out_zip.with_suffix('.zip')
print(f'Saved {zip_path} ({zip_path.stat().st_size / 1e6:.0f} MB)')
print('Download via the Jupyter file browser, or:')
print(f'  scp root@<vast-ip>:{zip_path} ./')

## 11 — Local setup after download

Unzip so the trained model lands where the local baseline expects it:

```bash
mkdir -p results/nnunet
unzip nnunet_model_Dataset001_NSCLCRadiomics.zip -d results/nnunet/
```

This yields
`results/nnunet/Dataset001_NSCLCRadiomics/nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres/`,
which is the default `--nnunet-model` path in `scripts/evaluation/run_baselines.py`.
Then re-run the sanity baseline to get real nnU-Net DSC:

```bash
python -m scripts.evaluation.run_baselines
```

(If you trained with the full `nnUNetTrainer` instead, pass the matching folder via
`--nnunet-model`.)